In [3]:
import time
import importlib
import pandas as pd
from pathlib import Path

import src.query as query_module
import src.product_search as product_search
import src.review_search as review_search
import src.evidence as evidence
import src.llm as llm
import src.comparison as comparison
import src.analytics as analytics
import src.utils as utils

In [4]:
importlib.reload(query_module)
importlib.reload(product_search)
importlib.reload(review_search)
importlib.reload(evidence)
importlib.reload(llm)
importlib.reload(comparison)
importlib.reload(analytics)
importlib.reload(utils)

from src.query import analyze_query, parse_query

from src.product_search import (
    build_product_index,
    search_products,
    filter_products
)

from src.review_search import (
    build_review_index,
    search_reviews
)

from src.evidence import (
    build_evidence,
    format_evidence_for_llm
)

from src.comparison import (
    compare_products
)

44 models reachable.


In [5]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

DATA_DIR = Path.cwd() / "data"

started = time.time()
products = pd.read_parquet(DATA_DIR / "products_work.parquet")
reviews = pd.read_parquet(
    DATA_DIR / "comments_work.parquet",
    columns=["id", "product_id", "body", "body_norm", "title", "rate",
             "recommendation_status", "is_buyer", "likes", "body_len",
             "is_substantive", "advantages", "disadvantages", "created_at"],
)
print(f"loaded in {time.time() - started:.1f}s")
print(f"products: {len(products):>9,} x {products.shape[1]}")
print(f"comments: {len(reviews):>9,} x {reviews.shape[1]}")

loaded in 15.7s
products:   331,599 x 33
comments: 6,153,060 x 14


In [6]:
start = time.perf_counter()

product_index = build_product_index(
    products
)

elapsed = (
    time.perf_counter()
    - start
) * 1000

print(
    "Product index shape:",
    product_index["matrix"].shape
)

print(
    "Build time:",
    round(elapsed, 2),
    "ms"
)

Product index shape: (331599, 100000)
Build time: 13636.13 ms


In [7]:
start = time.perf_counter()

review_index = build_review_index(
    reviews
)

elapsed = (
    time.perf_counter()
    - start
) * 1000

print(
    "Review index shape:",
    review_index["matrix"].shape
)

print(
    "Build time:",
    round(elapsed, 2),
    "ms"
)

print(
    "\nReview columns:"
)

print(
    review_index["columns"]
)

Review index shape: (6153060, 100000)
Build time: 197871.85 ms

Review columns:
{'product_id': 'product_id', 'review_id': 'id', 'text': 'body', 'rating': 'rate', 'recommendation': None, 'pros': 'advantages', 'cons': 'disadvantages'}


parse query

In [8]:
test_query = (
    "یک کیف برای استفاده روزمره میخوام "
    "که خیلی گرون نباشه و خریدارها هم "
    "ازش راضی باشن"
)

plan = parse_query(
    test_query
)

print("QUERY:")
print(test_query)

print("\nPARSED PLAN:")
print(plan)

QUERY:
یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن

PARSED PLAN:
{'query': 'یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن', 'concept': 'کیف', 'concept_terms': ['کیف', 'کیف دستی', 'کیف دوشی', 'کیف رودوشی', 'کیف زنانه'], 'brand': None, 'min_price': None, 'max_price': 5000000, 'require_satisfaction': True, 'sub_category': 'روزمره'}


Product search

In [148]:
query_text = (
    "یک کیف برای استفاده روزمره میخوام "
    "که خیلی گرون نباشه و خریدارها "
    "هم ازش راضی باشن"
)

plan = parse_query(
    query_text
)

start = time.perf_counter()

results = search_products(
    products=products,
    product_index=product_index,
    plan=plan,
    top_k=5
)

latency = (
    time.perf_counter() - start
) * 1000

print("QUERY:")
print(query_text)

print("\nPLAN:")
print(plan)

print(
    "\nRESULT COUNT:",
    len(results)
)

print(
    "LATENCY:",
    round(latency, 2),
    "ms"
)

display(
    results[
        [
            "id",
            "title_fa",
            "Price",
            "Brand",
            "mean_rate",
            "rec_ratio",
            "_satisfaction_score",
            "_final_score",
        ]
    ]
)

QUERY:
یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن

PLAN:
{'query': 'یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن', 'concept': 'کیف', 'concept_terms': ['کیف', 'کیف دستی', 'کیف دوشی', 'کیف رودوشی', 'کیف زنانه'], 'brand': None, 'min_price': None, 'max_price': 5000000, 'require_satisfaction': True, 'sub_category': 'روزمره'}

RESULT COUNT: 5
LATENCY: 9070.84 ms


,id,title_fa,Price,Brand,mean_rate,rec_ratio,_satisfaction_score,_final_score
50624,2006433,کیف زنانه کد 431405,2090000,متفرقه,4.246842,0.887701,0.864904,0.467704
201645,7397958,کیف رودوشی زنانه مدل 2376,1895000,متفرقه,4.263810,0.848837,0.838972,0.447987
318232,11674668,کیف رودوشی زنانه مدل 2872,2200000,متفرقه,4.571429,1.0,0.967857,0.444775
299664,10802035,کیف رودوشی زنانه مدل 2748,1490000,متفرقه,4.173333,0.854545,0.836182,0.442959
276352,9890431,کیف رودوشی زنانه مدل gckf09,2184000,متفرقه,4.750000,1.0,0.981250,0.439441


Review Search

In [149]:
product_id = 5715766

review_query = (
    "کاربران درباره کیفیت و عملکرد "
    "این محصول چه نظری دارند؟"
)

start = time.perf_counter()

review_results = search_reviews(
    query=review_query,
    product_id=product_id,
    reviews=reviews,
    review_index=review_index,
    top_k=8
)

latency = (
    time.perf_counter() - start
) * 1000

print(
    "PRODUCT ID:",
    product_id
)

print(
    "\nQUERY:",
    review_query
)

print(
    "\nREVIEWS:",
    len(review_results)
)

print(
    "LATENCY:",
    round(latency, 2),
    "ms"
)

display(
    review_results[
        [
            "id",
            "product_id",
            "body",
            "rate",
            "recommendation_status",
            "is_buyer",
            "_final_review_score",
        ]
    ]
)

PRODUCT ID: 5715766

QUERY: کاربران درباره کیفیت و عملکرد این محصول چه نظری دارند؟

REVIEWS: 8
LATENCY: 5577.37 ms


,id,product_id,body,rate,recommendation_status,is_buyer,_final_review_score
6139393,54589329,5715766,سبک و زود جذب,5.0,recommended,True,0.203523
6119353,54496305,5715766,پوشش دهی خوبی داشت,5.0,recommended,True,0.203434
6067628,54325088,5715766,خوبه برای استفاده روزانه ، بار اول دارم استفاده میکنم امیدوارم جوش...,5.0,recommended,True,0.192262
5984016,54075631,5715766,بدون بو\nاصلا چشم رو نمیسوزونه\nیکم رنگ پوستو روشن میکنه اما تو ذو...,5.0,recommended,True,0.190226
6111643,54467666,5715766,عالی عالیمن اولین بار که خریدم به پیشنهاد کاربران دیجی کالا ولی بع...,5.0,recommended,True,0.189430
6066259,54320932,5715766,عالی خیلی سبک و زود جذب,5.0,recommended,True,0.188032
6061867,54307635,5715766,پوست من با خیلی ضد آفتاب ها جوش زیر پوستی میزد اما با این فلوئید ا...,5.0,recommended,True,0.183379
5962581,54011628,5715766,باهاش جوش نمیزنم\r\nزود جذب میشه\r\nبرای بار دوم شارژ کردم,5.0,recommended,True,0.179920


Evidence

In [150]:
product_row = products[
    products["id"] == product_id
].iloc[0]

product_evidence = build_evidence(
    product=product_row,
    reviews=review_results,
    max_reviews=8
)

evidence_text = (
    format_evidence_for_llm(
        product_evidence
    )
)

print(evidence_text)

PRODUCT FACTS:
- ID: 5715766
- Title: فلوئید ضد آفتاب مای مدل Hyaluronic Acid حجم 50 میلی لیتر 
- Brand: مای
- Category: مراقبت پوست
- Sub-category: کرم ضد آفتاب
- Price: 1501000.0
- Rating: 4.400862216949463
- Rating count: 6538
- Recommendation ratio: 0.9074889867841409
- Review count: 233
- Buyer count: 227

REVIEW STATISTICS:
- count: 8
- average_rating: 5.0
- recommended_count: 8
- not_recommended_count: 0
- buyer_count: 8
- substantive_count: 5

REVIEW EVIDENCE:

[Evidence 1 | Review ID: 54325088]
Text: خوبه برای استفاده روزانه ، بار اول دارم استفاده میکنم امیدوارم جوش نزنم
Rating: 5.0
Recommendation: recommended
Advantages: راحتی استفاده
Disadvantages: حجم کم
Evidence quality: 0.300

[Evidence 2 | Review ID: 54075631]
Text: بدون بو
اصلا چشم رو نمیسوزونه
یکم رنگ پوستو روشن میکنه اما تو ذوق نمیزنه خیلی کم
باهاش جوش نزدم
به راحتی میشه تمدید کرد
بار دوم هستش ک میخرم
Rating: 5.0
Recommendation: recommended
Evidence quality: 0.300

[Evidence 3 | Review ID: 54467666]
Text: عالی عالیمن 

Review + LLM

In [154]:
question = (
    "ایرادهای پرتکرار این محصول چیست؟"
)

start = time.perf_counter()

llm_result = llm.call_llm(
    query=question,
    evidence_text=evidence_text,
    structured=False,
    use_cache=True
)

latency = (
    time.perf_counter() - start
) * 1000

print("ANSWER:\n")

print(
    llm_result["result"]["answer"]
)

print("\nUSAGE:")
print(
    llm_result["usage"]
)

print(
    "\nCACHE HIT:",
    llm_result["cache_hit"]
)

print(
    "\nLATENCY:",
    round(latency, 2),
    "ms"
)

ANSWER:

بر اساس شواهد موجود، تنها ایرادی که ذکر شده، حجم کم محصول است (Review ID: 54325088). هیچ ایراد پرتکرار دیگری در بررسی‌ها مشاهده نشد.

USAGE:
{'input_tokens': 1041, 'output_tokens': 272, 'total_tokens': 1313}

CACHE HIT: True

LATENCY: 0.74 ms


Comparison

In [156]:
product_ids = [
    11674668,
    7397958
]

comparison_result = compare_products(
    products=products,
    reviews=reviews,
    review_index=review_index,
    product_ids=product_ids,
    query=(
        "این دو محصول را از نظر "
        "تجربه کاربران و رضایت مقایسه کن"
    ),
    reviews_per_product=5
)

print("PRODUCTS:")

display(
    comparison_result["table"]
)

PRODUCTS:


,id,title_fa,Brand,Category1,Category2,Price,Rate,Rate_cnt,mean_rate,rec_ratio,n_reviews,n_buyers
201645,7397958,کیف رودوشی زنانه مدل 2376,متفرقه,اکسسوری زنانه,کیف زنانه,1895000,82,90,4.263810,0.848837,109,106
318232,11674668,کیف رودوشی زنانه مدل 2872,متفرقه,اکسسوری زنانه,کیف زنانه,2200000,80,2,4.571429,1.0,7,7


In [158]:
comparison_llm = llm.call_llm(
    query=(
        "این دو محصول را از نظر "
        "تجربه کاربران و رضایت مقایسه کن"
    ),
    evidence_text=(
        comparison_result[
            "llm_context"
        ]
    ),
    structured=False,
    use_cache=True
)

print(
    comparison_llm[
        "result"
    ]["answer"]
)

print(
    "\nUSAGE:",
    comparison_llm["usage"]
)

print(
    "CACHE:",
    comparison_llm["cache_hit"]
)

کیف رودوشی زنانه مدل 2872 (ID: 11674668) با امتیاز کلی 4.57 و نسبت توصیه 100%، رضایت بالاتری را نشان می‌دهد، در حالی که کیف رودوشی زنانه مدل 2376 (ID: 7397958) امتیاز کلی 4.26 و نسبت توصیه 84.88% دارد.

کاربران مدل 2872 آن را "خیلی عالی" برای گوشی و کلید، "کاملا سبک و زیبا" (Review ID: 53972817) و "خیلی قشنگ و کیوت با اندازه خوب" (Review ID: 52161213) توصیف کرده‌اند. در مقابل، کاربران مدل 2376 از آن برای استفاده دم دستی راضی بوده‌اند، آن را "سایز متوسط" و "بند نرم" (Review ID: 45945051) و "بسیار عالی، شیک، جادار و سبک" (Review ID: 45845999) دانسته‌اند.

لازم به ذکر است که مدل 2376 با 90 امتیاز و 109 نقد، بازخورد بسیار بیشتری نسبت به مدل 2872 با 2 امتیاز و 7 نقد دریافت کرده است.

USAGE: {'input_tokens': 1137, 'output_tokens': 1724, 'total_tokens': 2861}
CACHE: True


Analytics

In [159]:
analytics_result = analytics.run_analytics(
    products=products,
    reviews=reviews,
    query=(
        "پرتکرارترین شکایت کاربران چیست؟"
    )
)

display(
    analytics_result
)

{'intent': 'complaints',
 'result':               feature  count
 0           قیمت بالا  10049
 1                قیمت   4559
 2         کیفیت پایین   2909
 3         ماندگاری کم   2617
 4            بی کیفیت   2545
 5              حجم کم   2412
 6                نازک   1873
 7                گران   1696
 8              جنس بد   1446
 9            جنس ضعیف   1432
 10               کوچک   1401
 11              کیفیت   1185
 12             بوی بد   1155
 13  بسته بندی نامناسب   1002
 14                جنس    963
 15          قیمت زیاد    948
 16             کوچیکه    890
 17          سایز کوچک    868
 18          بسته بندی    815
 19              کوچیک    787}

In [160]:
analytics_llm = llm.call_llm(
    query=(
        "پرتکرارترین شکایت کاربران چیست؟"
    ),
    evidence_text=str(
        analytics_result
    ),
    structured=False,
    use_cache=True
)

print(
    analytics_llm[
        "result"
    ]["answer"]
)

print(
    "\nUSAGE:",
    analytics_llm["usage"]
)

پرتکرارترین شکایت کاربران "قیمت بالا" است که ۱۰,۰۴۹ بار ذکر شده است.

USAGE: {'input_tokens': 443, 'output_tokens': 89, 'total_tokens': 532}


In [161]:
query_text = (
    "فلوئید ضد آفتاب مای مدل "
    "Hyaluronic Acid کیفیتش چطوره؟"
)

print("FIRST CALL")

r1 = llm.call_llm(
    query=query_text,
    evidence_text=evidence_text,
    structured=False,
    use_cache=True
)

print(
    "Cache:",
    r1["cache_hit"]
)

print(
    "Usage:",
    r1["usage"]
)


print("\nSECOND CALL")

r2 = llm.call_llm(
    query=query_text,
    evidence_text=evidence_text,
    structured=False,
    use_cache=True
)

print(
    "Cache:",
    r2["cache_hit"]
)

print(
    "Usage:",
    r2["usage"]
)

FIRST CALL
Cache: False
Usage: {'input_tokens': 1049, 'output_tokens': 886, 'total_tokens': 1935}

SECOND CALL
Cache: True
Usage: {'input_tokens': 1049, 'output_tokens': 886, 'total_tokens': 1935}


In [162]:
query_text = (
    "یک کیف برای استفاده روزمره میخوام "
    "که خیلی گرون نباشه و خریدارها "
    "هم ازش راضی باشن"
)

print("=" * 80)
print("USER:")
print(query_text)
print("=" * 80)

plan = parse_query(
    query_text
)

print("\nPARSED QUERY:")
print(plan)

start = time.perf_counter()

results = search_products(
    products=products,
    product_index=product_index,
    plan=plan,
    top_k=5
)

latency = (
    time.perf_counter() - start
) * 1000

print("\nRESULTS:")
print(
    format_search_answer(
        results
    )
)

print(
    "\nLATENCY:",
    round(latency, 2),
    "ms"
)

print(
    "API CALL:",
    "No"
)

USER:
یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن

PARSED QUERY:
{'query': 'یک کیف برای استفاده روزمره میخوام که خیلی گرون نباشه و خریدارها هم ازش راضی باشن', 'concept': 'کیف', 'concept_terms': ['کیف', 'کیف دستی', 'کیف دوشی', 'کیف رودوشی', 'کیف زنانه'], 'brand': None, 'min_price': None, 'max_price': 5000000, 'require_satisfaction': True, 'sub_category': 'روزمره'}

RESULTS:
5 محصول مرتبط پیدا شد:

1. کیف زنانه کد 431405
   قیمت: 2,090,000 تومان
   امتیاز: 4.25/5
   پیشنهاد خرید: 88.8%

2. کیف رودوشی زنانه مدل 2376
   قیمت: 1,895,000 تومان
   امتیاز: 4.26/5
   پیشنهاد خرید: 84.9%

3. کیف رودوشی زنانه مدل 2872
   قیمت: 2,200,000 تومان
   امتیاز: 4.57/5
   پیشنهاد خرید: 100.0%

4. کیف رودوشی زنانه مدل 2748
   قیمت: 1,490,000 تومان
   امتیاز: 4.17/5
   پیشنهاد خرید: 85.5%

5. کیف رودوشی زنانه مدل gckf09
   قیمت: 2,184,000 تومان
   امتیاز: 4.75/5
   پیشنهاد خرید: 100.0%

LATENCY: 2960.25 ms
API CALL: No


execute query

In [9]:
def execute_query(
    query_text,
    top_k_products=5,
    top_k_reviews=5,
    use_cache=True,
):
    """
    Execute a user query directly.

    The user only provides natural-language text.
    """

    query_text = str(
        query_text
    ).strip()

    if not query_text:
        return {
            "success": False,
            "error": "Query is empty."
        }

    # --------------------------------------------------------
    # Parse query
    # --------------------------------------------------------

    plan = parse_query(
        query_text
    )

    print("=" * 80)
    print("USER QUERY:")
    print(query_text)

    print("\nPARSED QUERY:")
    print(plan)

    # --------------------------------------------------------
    # Current design:
    #
    # We determine the requested operation from the
    # information in the query itself.
    # --------------------------------------------------------

    query_lower = query_text.lower()

    # --------------------------------------------------------
    # Comparison
    # --------------------------------------------------------

    comparison_signals = [
        "مقایسه",
        "مقایسه کن",
        "مقایسه کنید",
        "با هم مقایسه",
        "کدوم بهتر",
        "کدام بهتر",
        "تفاوت",
        "فرق",
    ]

    is_comparison = any(
        signal in query_lower
        for signal in comparison_signals
    )

    # --------------------------------------------------------
    # Review
    # --------------------------------------------------------

    review_signals = [
        "نظر",
        "نظرات",
        "بازخورد",
        "تجربه",
        "ایراد",
        "مشکل",
        "کیفیت",
        "دوام",
        "جنس",
        "رضایت",
        "ارزش خرید",
    ]

    is_review = any(
        signal in query_lower
        for signal in review_signals
    )

    # --------------------------------------------------------
    # Analytics
    # --------------------------------------------------------

    analytics_signals = [
        "تحلیل",
        "آمار",
        "گزارش",
        "پرتکرارترین",
        "شکایت",
        "دسته",
        "برند",
    ]

    is_analytics = any(
        signal in query_lower
        for signal in analytics_signals
    )

    # --------------------------------------------------------
    # Analytics
    # --------------------------------------------------------

    if is_analytics:

        analysis = analytics.run_analytics(
            products=products,
            reviews=reviews,
            query=query_text,
        )

        llm_result = llm.call_llm(
            query=query_text,
            evidence_text=str(
                analysis
            ),
            structured=False,
            use_cache=use_cache,
        )

        return {
            "success": True,
            "type": "analytics",
            "plan": plan,
            "data": analysis,
            "answer": llm_result["result"],
            "usage": llm_result["usage"],
            "cache_hit": llm_result["cache_hit"],
        }

    # ========================================================
    # Comparison
    # ========================================================

    if is_comparison:

        resolution = (
            resolve_products_from_query(
                query_text
            )
        )

        if (
            not resolution["resolved"]
            or len(
                resolution["product_ids"]
            ) < 2
        ):

            return {
                "success": False,
                "type": "comparison",
                "plan": plan,
                "resolution": resolution,
                "message": (
                    "لطفاً نام یا شناسه دو محصول "
                    "را داخل Query مشخص کنید."
                ),
            }

        product_ids = resolution[
            "product_ids"
        ][:2]

        comparison_result = (
            compare_products(
                products=products,
                reviews=reviews,
                review_index=review_index,
                product_ids=product_ids,
                query=query_text,
                reviews_per_product=top_k_reviews,
            )
        )

        llm_result = llm.call_llm(
            query=query_text,
            evidence_text=(
                comparison_result[
                    "llm_context"
                ]
            ),
            structured=False,
            use_cache=use_cache,
        )

        return {
            "success": True,
            "type": "comparison",
            "plan": plan,
            "resolution": resolution,
            "products": comparison_result[
                "table"
            ],
            "evidence": comparison_result[
                "evidence"
            ],
            "evidence_text": comparison_result[
                "llm_context"
            ],
            "answer": llm_result["result"],
            "usage": llm_result["usage"],
            "cache_hit": llm_result[
                "cache_hit"
            ],
        }

    # ========================================================
    # Review
    # ========================================================

    if is_review:

        resolution = (
            resolve_products_from_query(
                query_text
            )
        )

        if not resolution["resolved"]:

            return {
                "success": False,
                "type": "review",
                "plan": plan,
                "resolution": resolution,
                "message": (
                    "لطفاً نام یا شناسه محصول "
                    "را داخل Query مشخص کنید."
                ),
            }

        product_id = resolution[
            "product_ids"
        ][0]

        product_rows = products[
            products["id"] == product_id
        ]

        if product_rows.empty:

            return {
                "success": False,
                "type": "review",
                "message": (
                    "محصول موردنظر پیدا نشد."
                ),
            }

        product_row = (
            product_rows.iloc[0]
        )

        retrieved_reviews = search_reviews(
            query=query_text,
            product_id=product_id,
            reviews=reviews,
            review_index=review_index,
            top_k=top_k_reviews,
        )

        product_evidence = build_evidence(
            product=product_row,
            reviews=retrieved_reviews,
            max_reviews=top_k_reviews,
        )

        evidence_text = (
            format_evidence_for_llm(
                product_evidence
            )
        )

        llm_result = llm.call_llm(
            query=query_text,
            evidence_text=evidence_text,
            structured=False,
            use_cache=use_cache,
        )

        return {
            "success": True,
            "type": "review",
            "plan": plan,
            "resolution": resolution,
            "product": product_row,
            "reviews": retrieved_reviews,
            "evidence": product_evidence,
            "evidence_text": evidence_text,
            "answer": llm_result["result"],
            "usage": llm_result["usage"],
            "cache_hit": llm_result[
                "cache_hit"
            ],
        }

    # ========================================================
    # Product Search
    # ========================================================

    product_results = search_products(
        products=products,
        product_index=product_index,
        plan=plan,
        top_k=top_k_products,
    )

    return {
        "success": True,
        "type": "search",
        "plan": plan,
        "products": product_results,
        "answer": {
            "answer": (
                format_search_answer(
                    product_results
                )
            )
        },
        "usage": {
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
        },
        "cache_hit": False,
    }

In [11]:
import re
from rapidfuzz import fuzz, process


def extract_ids_from_query(
    query
):
    """
    Extract numeric product IDs written in query.
    """

    numbers = re.findall(
        r"\b\d{4,12}\b",
        str(query)
    )

    if not numbers:
        return []

    valid_ids = set(
        pd.to_numeric(
            products["id"],
            errors="coerce"
        )
        .dropna()
        .astype(int)
        .tolist()
    )

    ids = []

    for number in numbers:

        product_id = int(number)

        if product_id in valid_ids:

            ids.append(
                product_id
            )

    return list(
        dict.fromkeys(
            ids
        )
    )


def find_products_by_name(
    query,
    top_k=5,
    min_score=70,
):
    """
    Find products mentioned by name in a natural-language query.
    """

    if "title_norm" not in products.columns:
        return []

    query_norm = normalize_persian_text(
        str(query)
    ).strip().lower()

    # Remove common question phrases.
    removable = [
        "کیفیتش چطوره",
        "کیفیتش چگونه است",
        "کیفیتش خوبه",
        "نظرات کاربران درباره",
        "نظر کاربران درباره",
        "نظرات خریداران درباره",
        "نظر خریداران درباره",
        "ایرادهای این محصول چیه",
        "ایرادهای این محصول چیست",
        "مشکلات این محصول چیه",
        "مشکلات این محصول چیست",
        "بررسی کن",
        "بررسیش کن",
        "مقایسه کن",
        "مقایسه کنید",
        "رو مقایسه کن",
        "را مقایسه کن",
        "از نظر",
        "کدام بهتر است",
        "کدوم بهتره",
    ]

    for phrase in removable:

        query_norm = query_norm.replace(
            normalize_persian_text(
                phrase
            ),
            " "
        )

    # Remove product IDs.
    query_norm = re.sub(
        r"\b\d{4,12}\b",
        " ",
        query_norm
    )

    # Remove generic words.
    generic_words = [
        "محصول",
        "کالا",
        "این",
        "این محصول",
        "این کالا",
        "دو",
        "را",
        "رو",
        "کن",
        "با",
        "هم",
        "و",
    ]

    for word in generic_words:

        query_norm = query_norm.replace(
            word,
            " "
        )

    query_norm = re.sub(
        r"\s+",
        " ",
        query_norm
    ).strip()

    if len(query_norm) < 3:
        return []

    titles = (
        products["title_norm"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    # --------------------------------------------------------
    # Exact containment
    # --------------------------------------------------------

    mask = titles.str.contains(
        re.escape(query_norm),
        regex=True,
        na=False
    )

    exact_matches = products.loc[
        mask
    ]

    if not exact_matches.empty:

        return [
            {
                "id": int(row["id"]),
                "title": row["title_fa"],
                "score": 100.0,
                "match_type": "exact",
            }
            for _, row
            in exact_matches.head(
                top_k
            ).iterrows()
        ]

    # --------------------------------------------------------
    # Token-overlap candidates
    # --------------------------------------------------------

    query_tokens = {
        token
        for token in query_norm.split()
        if len(token) >= 2
    }

    candidates = []

    for token in query_tokens:

        mask = titles.str.contains(
            re.escape(token),
            regex=True,
            na=False
        )

        token_matches = products.loc[
            mask,
            [
                "id",
                "title_fa",
                "title_norm",
            ]
        ]

        candidates.append(
            token_matches
        )

    if candidates:

        candidates = pd.concat(
            candidates,
            ignore_index=False
        ).drop_duplicates(
            subset=["id"]
        )

        scored = []

        for _, row in candidates.iterrows():

            title_tokens = {
                token
                for token
                in str(
                    row["title_norm"]
                ).split()
                if len(token) >= 2
            }

            if not title_tokens:
                continue

            overlap = (
                len(
                    query_tokens
                    & title_tokens
                )
                /
                len(query_tokens)
            )

            scored.append({
                "id": int(row["id"]),
                "title": row["title_fa"],
                "score": overlap * 100,
                "match_type": "token",
            })

        scored.sort(
            key=lambda x: x["score"],
            reverse=True
        )

        if scored:
            return [
                item
                for item in scored[:top_k]
                if item["score"] >= min_score
            ]

    return []


def resolve_products_from_query(
    query
):
    """
    Resolve products directly from the user's query.

    Priority:
        1. Explicit product IDs
        2. Product names

    No product is guessed.
    """

    # --------------------------------------------------------
    # IDs
    # --------------------------------------------------------

    ids = extract_ids_from_query(
        query
    )

    if ids:

        return {
            "resolved": True,
            "source": "query_id",
            "product_ids": ids,
            "confidence": 1.0,
        }

    # --------------------------------------------------------
    # Names
    # --------------------------------------------------------

    matches = find_products_by_name(
        query
    )

    if matches:

        return {
            "resolved": True,
            "source": "query_name",
            "product_ids": [
                item["id"]
                for item in matches
            ],
            "confidence": (
                matches[0]["score"]
                / 100
            ),
            "candidates": matches,
        }

    return {
        "resolved": False,
        "source": None,
        "product_ids": [],
        "confidence": 0.0,
        "candidates": [],
    }

Test Query

In [166]:
result = execute_query(
    "یک کیف زنانه زیر ۲ میلیون میخوام"
)

print(
    result["answer"]["answer"]
)

USER QUERY:
یک کیف زنانه زیر ۲ میلیون میخوام

PARSED QUERY:
{'query': 'یک کیف زنانه زیر 2 میلیون میخوام', 'concept': 'کیف', 'concept_terms': ['کیف', 'کیف دستی', 'کیف دوشی', 'کیف رودوشی', 'کیف زنانه'], 'brand': None, 'min_price': None, 'max_price': 2000000.0, 'require_satisfaction': False, 'sub_category': 'زنانه'}
5 محصول مرتبط پیدا شد:

1. آویز کیف زنانه کد Fr710
   قیمت: 420,000 تومان
   امتیاز: 4.75/5
   پیشنهاد خرید: 100.0%

2. آویز کیف زنانه کد Fr735
   قیمت: 370,000 تومان
   امتیاز: 4.20/5
   پیشنهاد خرید: 100.0%

3. آویز کیف زنانه مدل KH1013
   قیمت: 787,500 تومان
   امتیاز: 5.00/5
   پیشنهاد خرید: 100.0%

4. کاور کیف زنانه کد M01
   قیمت: 357,500 تومان
   امتیاز: 4.00/5
   پیشنهاد خرید: 100.0%

5. آویز کیف زنانه مدل KH1007
   قیمت: 742,500 تومان
   امتیاز: 4.00/5
   پیشنهاد خرید: 100.0%


e:\Quera_Proj_3\src\product_search.py:1069: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<ArrowExtensionArray>
[0.15000000596046448,  0.7916666756073634,  0.6822916696468988,  0.5750000089406967,   1.000000011920929,  0.9250000089406967,  0.8519230838005359,  0.9399999946355819,             0.98125,
  0.7215086032867432,    0.81114334874558,  0.5000000059604645,  0.7440909084948626,  0.8279962291804756,  0.7564193710088729,  0.9250000089406967, 0.15000000596046448,  0.9625000178813934,
 0.10500001162290573, 0.07500000298023224,  0.8361818087642843,                 0.0,  0.7076830705980849, 0.27500000536441804,  0.4020833363135655,  0.8500000059604644,  0.6761489384351893,
  0.8134095647850552,  0.7100000083446503,  0.9437500059604644,  0.9250000089406967,   1.000000011920929,              0.5375,  0.7214285731315613,  0.6144097252024544,  0.5399999973177909,
  0.8208333303531011,  0.5562500119209289,   

Review با ID

In [13]:
result = execute_query(
    "محصول 5715766 ایرادهای پرتکرارش چیه؟"
)

print(
    result["answer"]["answer"]
)

USER QUERY:
محصول 5715766 ایرادهای پرتکرارش چیه؟

PARSED QUERY:
{'query': 'محصول 5715766 ایرادهای پرتکرارش چیه؟', 'concept': None, 'concept_terms': [], 'brand': None, 'min_price': None, 'max_price': None, 'require_satisfaction': False, 'sub_category': None}
بر اساس شواهد موجود، هیچ ایراد پرتکراری برای محصول 5715766 ذکر نشده است. تمامی نظرات ارائه شده (شواهد 1 تا 5) این محصول را مثبت ارزیابی کرده‌اند. به عنوان مثال، کاربران آن را سبک، زود جذب، بدون سفیدی و مناسب پوست چرب توصیف کرده‌اند.


Comparison با دو ID

In [14]:
result = execute_query(
    "محصولات 5715766 و 9671023 رو از نظر رضایت کاربران مقایسه کن"
)

print(
    result["answer"]["answer"]
)

USER QUERY:
محصولات 5715766 و 9671023 رو از نظر رضایت کاربران مقایسه کن

PARSED QUERY:
{'query': 'محصولات 5715766 و 9671023 رو از نظر رضایت کاربران مقایسه کن', 'concept': None, 'concept_terms': [], 'brand': None, 'min_price': None, 'max_price': None, 'require_satisfaction': True, 'sub_category': None}
محصول 5715766 (فلوئید ضد آفتاب مای) با امتیاز کلی 4.4 و 233 نقد، رضایت گسترده‌تری را نشان می‌دهد، در حالی که محصول 9671023 (کیف دوشی زنانه) با امتیاز کلی 4.3 و 20 نقد، تعداد کاربران کمتری دارد. هر دو محصول در بررسی‌های ارائه شده، میانگین امتیاز 4.8 و 100% توصیه را کسب کرده‌اند. کاربران محصول 5715766 از جذب سریع، سبک بودن و بی‌رنگ بودن آن راضی هستند (شناسه نقد: 54012338، 54463043، 54045322)، اما برخی به حجم کم آن اشاره کرده‌اند (شناسه نقد: 54012338، 54045322). کاربران محصول 9671023 از جادار بودن، طراحی زیبا و کیفیت ساخت آن رضایت دارند (شناسه نقد: 53063283، 47567116، 47329044) و هیچ نکته منفی در بررسی‌های ارائه شده برای آن ذکر نشده است.


ساخت دیتای برای ارزیابی

In [15]:
import pandas as pd
import numpy as np


def sample_evaluation_data(
    products,
    reviews,
    n_products=100,
    n_reviews=100,
    random_state=42,
):
    """
    Extract diverse real samples from the prepared dataset
    for building an evaluation set.

    The function does NOT create gold labels.
    It only creates candidate samples for manual annotation.
    """

    rng = np.random.default_rng(
        random_state
    )

    # ========================================================
    # 1. PRODUCT SAMPLES
    # ========================================================

    product_samples = []

    # --------------------------------------------------------
    # A. Products with many reviews
    # --------------------------------------------------------

    if "n_reviews" in products.columns:

        high_review = (
            products
            .sort_values(
                "n_reviews",
                ascending=False
            )
            .head(
                n_products // 4
            )
        )

        product_samples.append(
            high_review
        )

    # --------------------------------------------------------
    # B. Highly rated products
    # --------------------------------------------------------

    if "mean_rate" in products.columns:

        high_rating = (
            products[
                products["mean_rate"].notna()
            ]
            .sort_values(
                "mean_rate",
                ascending=False
            )
            .head(
                n_products // 4
            )
        )

        product_samples.append(
            high_rating
        )

    # --------------------------------------------------------
    # C. Low rated / problematic products
    # --------------------------------------------------------

    if "mean_rate" in products.columns:

        low_rating = (
            products[
                products["mean_rate"].notna()
            ]
            .sort_values(
                "mean_rate",
                ascending=True
            )
            .head(
                n_products // 4
            )
        )

        product_samples.append(
            low_rating
        )

    # --------------------------------------------------------
    # D. Random products
    # --------------------------------------------------------

    remaining_n = (
        n_products
        - sum(
            len(x)
            for x in product_samples
        )
    )

    if remaining_n > 0:

        random_products = (
            products
            .sample(
                n=min(
                    remaining_n,
                    len(products)
                ),
                random_state=random_state
            )
        )

        product_samples.append(
            random_products
        )

    product_samples = (
        pd.concat(
            product_samples,
            ignore_index=True
        )
        .drop_duplicates(
            subset=["id"]
        )
    )

    # --------------------------------------------------------
    # Limit
    # --------------------------------------------------------

    if len(product_samples) > n_products:

        product_samples = (
            product_samples
            .sample(
                n=n_products,
                random_state=random_state
            )
        )

    # ========================================================
    # 2. REVIEW SAMPLES
    # ========================================================

    review_samples = []

    # --------------------------------------------------------
    # A. Positive reviews
    # --------------------------------------------------------

    positive = reviews[
        (
            reviews["rate"]
            >= 4
        )
        &
        (
            reviews["body"]
            .notna()
        )
    ]

    if not positive.empty:

        review_samples.append(
            positive.sample(
                n=min(
                    n_reviews // 4,
                    len(positive)
                ),
                random_state=random_state
            )
        )

    # --------------------------------------------------------
    # B. Negative reviews
    # --------------------------------------------------------

    negative = reviews[
        (
            reviews["rate"]
            <= 2
        )
        &
        (
            reviews["body"]
            .notna()
        )
    ]

    if not negative.empty:

        review_samples.append(
            negative.sample(
                n=min(
                    n_reviews // 4,
                    len(negative)
                ),
                random_state=random_state
            )
        )

    # --------------------------------------------------------
    # C. Substantive reviews
    # --------------------------------------------------------

    if "is_substantive" in reviews.columns:

        substantive = reviews[
            (
                reviews["is_substantive"]
                == True
            )
            &
            (
                reviews["body"]
                .notna()
            )
        ]

        if not substantive.empty:

            review_samples.append(
                substantive.sample(
                    n=min(
                        n_reviews // 4,
                        len(substantive)
                    ),
                    random_state=random_state
                )
            )

    # --------------------------------------------------------
    # D. Random reviews
    # --------------------------------------------------------

    current_count = sum(
        len(x)
        for x in review_samples
    )

    remaining_n = (
        n_reviews
        - current_count
    )

    if remaining_n > 0:

        available_reviews = (
            reviews[
                reviews["body"].notna()
            ]
        )

        review_samples.append(
            available_reviews.sample(
                n=min(
                    remaining_n,
                    len(available_reviews)
                ),
                random_state=random_state
            )
        )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    review_samples = (
        pd.concat(
            review_samples,
            ignore_index=True
        )
        .drop_duplicates(
            subset=["id"]
        )
    )

    if len(review_samples) > n_reviews:

        review_samples = (
            review_samples
            .sample(
                n=n_reviews,
                random_state=random_state
            )
        )

    # ========================================================
    # 3. Return
    # ========================================================

    return {
        "products": product_samples.reset_index(
            drop=True
        ),

        "reviews": review_samples.reset_index(
            drop=True
        ),
    }

In [16]:
evaluation_samples = sample_evaluation_data(
    products=products,
    reviews=reviews,
    n_products=100,
    n_reviews=100,
    random_state=42,
)
evaluation_products = (
    evaluation_samples["products"]
)

evaluation_reviews = (
    evaluation_samples["reviews"]
)

print(
    "Products:",
    evaluation_products.shape
)

print(
    "Reviews:",
    evaluation_reviews.shape
)

Products: (100, 33)
Reviews: (100, 14)


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

EVAL_DIR = Path.cwd() / 'evaluation'

RANDOM_STATE = 42

N_SEARCH_QUERIES = 30
N_REVIEW_PRODUCTS = 15
REVIEWS_PER_PRODUCT = 8
SEARCH_CANDIDATES_PER_QUERY = 10
REVIEW_CANDIDATES_PER_QUERY = 10
N_COMPARISON_PAIRS = 15


# ============================================================
# LOAD DATA
# ============================================================

# products = pd.read_parquet(
#     PRODUCTS_PATH
# )

# reviews = pd.read_parquet(
#     REVIEWS_PATH
# )

print("Products:", products.shape)
print("Reviews :", reviews.shape)


EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# HELPERS
# ============================================================

def safe_text(
    value,
    default=None
):
    if pd.isna(value):
        return default

    value = str(value).strip()

    if not value:
        return default

    return value


# ============================================================
# 1. SELECT EVALUATION PRODUCTS
# ============================================================

def select_evaluation_products(
    products,
    n_products=30,
    min_reviews=10,
    random_state=42,
):
    df = products.copy()

    df["n_reviews_num"] = pd.to_numeric(
        df["n_reviews"],
        errors="coerce"
    ).fillna(0)

    df = df[
        df["n_reviews_num"] >= min_reviews
    ].copy()

    if df.empty:
        return df

    selected = []

    if "Category1" in df.columns:

        categories = (
            df["Category1"]
            .dropna()
            .unique()
        )

        per_category = max(
            1,
            n_products // max(
                len(categories),
                1
            )
        )

        for category in categories:

            group = df[
                df["Category1"] == category
            ]

            if group.empty:
                continue

            # High-review products
            high_review = (
                group
                .sort_values(
                    "n_reviews_num",
                    ascending=False
                )
                .head(
                    per_category
                )
            )

            selected.append(
                high_review
            )

    if selected:

        result = (
            pd.concat(
                selected,
                ignore_index=True
            )
            .drop_duplicates(
                subset=["id"]
            )
        )

    else:

        result = df.sample(
            n=min(
                n_products,
                len(df)
            ),
            random_state=random_state
        )

    # Fill remaining slots randomly
    if len(result) < n_products:

        remaining = df[
            ~df["id"].isin(
                result["id"]
            )
        ]

        if not remaining.empty:

            extra = remaining.sample(
                n=min(
                    n_products - len(result),
                    len(remaining)
                ),
                random_state=random_state
            )

            result = pd.concat(
                [
                    result,
                    extra
                ],
                ignore_index=True
            )

    return (
        result
        .drop_duplicates(
            subset=["id"]
        )
        .head(n_products)
        .reset_index(drop=True)
    )


eval_products = select_evaluation_products(
    products,
    n_products=N_SEARCH_QUERIES,
    min_reviews=10,
    random_state=RANDOM_STATE
)


# ============================================================
# 2. SELECT EVALUATION REVIEWS
# ============================================================

def select_evaluation_reviews(
    eval_products,
    reviews,
    reviews_per_product=8,
    random_state=42,
):
    product_ids = set(
        eval_products["id"]
        .astype(int)
        .tolist()
    )

    df = reviews[
        reviews["product_id"]
        .isin(product_ids)
    ].copy()

    selected = []

    for product_id in product_ids:

        group = df[
            df["product_id"] == product_id
        ].copy()

        if group.empty:
            continue

        parts = []

        # Positive
        positive = group[
            pd.to_numeric(
                group["rate"],
                errors="coerce"
            ) >= 4
        ]

        if not positive.empty:
            parts.append(
                positive.head(
                    max(
                        1,
                        reviews_per_product // 3
                    )
                )
            )

        # Negative
        negative = group[
            pd.to_numeric(
                group["rate"],
                errors="coerce"
            ) <= 2
        ]

        if not negative.empty:
            parts.append(
                negative.head(
                    max(
                        1,
                        reviews_per_product // 3
                    )
                )
            )

        # Substantive
        if "is_substantive" in group.columns:

            substantive = group[
                group["is_substantive"] == True
            ]

            if not substantive.empty:
                parts.append(
                    substantive.head(
                        max(
                            1,
                            reviews_per_product // 3
                        )
                    )
                )

        used_ids = set()

        for part in parts:
            if "id" in part.columns:
                used_ids.update(
                    part["id"].tolist()
                )

        current_count = sum(
            len(part)
            for part in parts
        )

        remaining = (
            reviews_per_product
            - current_count
        )

        if remaining > 0:

            available = group[
                ~group["id"].isin(
                    used_ids
                )
            ]

            if not available.empty:

                parts.append(
                    available.sample(
                        n=min(
                            remaining,
                            len(available)
                        ),
                        random_state=random_state
                    )
                )

        if parts:

            product_reviews = (
                pd.concat(
                    parts,
                    ignore_index=True
                )
                .drop_duplicates(
                    subset=["id"]
                )
                .head(
                    reviews_per_product
                )
            )

            selected.append(
                product_reviews
            )

    if not selected:

        return reviews.iloc[
            0:0
        ].copy()

    return pd.concat(
        selected,
        ignore_index=True
    )


eval_reviews = select_evaluation_reviews(
    eval_products,
    reviews,
    reviews_per_product=REVIEWS_PER_PRODUCT,
    random_state=RANDOM_STATE
)


# ============================================================
# 3. SEARCH QUERY CANDIDATES
# ============================================================

def build_search_queries(
    eval_products,
):
    rows = []

    for _, row in eval_products.iterrows():

        product_id = int(
            row["id"]
        )

        category = safe_text(
            row.get("Category2")
        )

        category1 = safe_text(
            row.get("Category1")
        )

        brand = safe_text(
            row.get("Brand")
        )

        price = row.get(
            "Price"
        )

        rec_ratio = row.get(
            "rec_ratio"
        )

        product_type = (
            category
            or category1
            or "محصول"
        )

        # ----------------------------------------------------
        # Category
        # ----------------------------------------------------

        rows.append({
            "query_id":
                f"search_{product_id}_category",

            "query":
                f"یک {product_type} میخوام",

            "target_product_id":
                product_id,

            "query_type":
                "category",
        })

        # ----------------------------------------------------
        # Brand
        # ----------------------------------------------------

        if (
            brand
            and brand != "متفرقه"
        ):

            rows.append({
                "query_id":
                    f"search_{product_id}_brand",

                "query":
                    f"یک محصول از برند {brand} میخوام",

                "target_product_id":
                    product_id,

                "query_type":
                    "brand",
            })

        # ----------------------------------------------------
        # Price
        # ----------------------------------------------------

        if pd.notna(price):

            try:
                price = float(price)
            except (
                ValueError,
                TypeError
            ):
                price = None

        if price is not None:

            upper = int(
                round(
                    price * 1.2,
                    -4
                )
            )

            rows.append({
                "query_id":
                    f"search_{product_id}_price",

                "query":
                    f"یک {product_type} "
                    f"زیر {upper:,} تومان میخوام",

                "target_product_id":
                    product_id,

                "query_type":
                    "price",
            })

        # ----------------------------------------------------
        # Satisfaction
        # ----------------------------------------------------

        if pd.notna(rec_ratio):

            try:
                rec_ratio = float(
                    rec_ratio
                )
            except (
                ValueError,
                TypeError
            ):
                rec_ratio = None

        if (
            rec_ratio is not None
            and rec_ratio >= 0.75
        ):

            rows.append({
                "query_id":
                    f"search_{product_id}_satisfaction",

                "query":
                    f"یک {product_type} "
                    "با رضایت بالای کاربران میخوام",

                "target_product_id":
                    product_id,

                "query_type":
                    "satisfaction",
            })

    return pd.DataFrame(
        rows
    )


search_candidates = build_search_queries(
    eval_products
)


# Select a balanced subset
search_queries = (
    search_candidates
    .groupby(
        "query_type",
        group_keys=False
    )
    .apply(
        lambda x:
            x.sample(
                n=min(
                    max(
                        1,
                        N_SEARCH_QUERIES // 4
                    ),
                    len(x)
                ),
                random_state=RANDOM_STATE
            ),
        include_groups=False
    )
    .reset_index(drop=True)
)

if len(search_queries) > N_SEARCH_QUERIES:

    search_queries = (
        search_queries
        .sample(
            n=N_SEARCH_QUERIES,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )


# ============================================================
# 4. REVIEW QUERY CANDIDATES
# ============================================================

review_templates = [
    "کاربران درباره کیفیت این محصول چه گفتند؟",
    "ایرادهای این محصول چیست؟",
    "مزیت‌های این محصول از نظر خریداران چیست؟",
    "کاربران درباره دوام این محصول چه گفتند؟",
    "خریداران از چه چیزی درباره این محصول ناراضی بودند؟",
    "ارزش خرید این محصول از نظر کاربران چطور است؟",
]


def build_review_queries(
    eval_products
):
    rows = []

    for _, row in eval_products.iterrows():

        product_id = int(
            row["id"]
        )

        title = row[
            "title_fa"
        ]

        for i, template in enumerate(
            review_templates,
            start=1
        ):

            rows.append({
                "query_id":
                    f"review_{product_id}_{i}",

                "product_id":
                    product_id,

                "product_title":
                    title,

                "query":
                    template,

                "query_type":
                    "review",
            })

    return pd.DataFrame(
        rows
    )


review_queries = build_review_queries(
    eval_products
)


# ============================================================
# 5. REVIEW CANDIDATES FOR ANNOTATION
# ============================================================

# NOTE:
# We retrieve candidates only after indexes have been built.
# If this script is executed standalone, build the review index
# with the project's existing function.

try:
    from src.review_search import (
        build_review_index,
        search_reviews,
    )

    review_index = build_review_index(
        reviews
    )

    review_annotation_rows = []

    for _, item in review_queries.iterrows():

        product_id = int(
            item["product_id"]
        )

        result = search_reviews(
            query=item["query"],
            product_id=product_id,
            reviews=reviews,
            review_index=review_index,
            top_k=REVIEW_CANDIDATES_PER_QUERY,
        )

        if result.empty:
            continue

        for rank, (_, review) in enumerate(
            result.iterrows(),
            start=1
        ):

            review_annotation_rows.append({
                "query_id":
                    item["query_id"],

                "query":
                    item["query"],

                "product_id":
                    product_id,

                "rank":
                    rank,

                "review_id":
                    int(review["id"]),

                "review_text":
                    review["body"],

                "rating":
                    review.get("rate"),

                "recommendation_status":
                    review.get(
                        "recommendation_status"
                    ),

                "is_buyer":
                    review.get(
                        "is_buyer"
                    ),

                "is_substantive":
                    review.get(
                        "is_substantive"
                    ),

                "retrieval_score":
                    review.get(
                        "_final_review_score"
                    ),

                "relevant":
                    None,

                "annotation_note":
                    "",
            })

    review_annotation = pd.DataFrame(
        review_annotation_rows
    )

except Exception as error:

    print(
        "Review candidate generation skipped:",
        error
    )

    review_annotation = pd.DataFrame()


# ============================================================
# 6. SEARCH CANDIDATES FOR ANNOTATION
# ============================================================

try:
    from src.product_search import (
        build_product_index,
        search_products,
    )

    product_index = build_product_index(
        products
    )

    search_annotation_rows = []

    for _, item in search_queries.iterrows():

        plan = parse_query(
            item["query"]
        )

        result = search_products(
            products=products,
            product_index=product_index,
            plan=plan,
            top_k=SEARCH_CANDIDATES_PER_QUERY,
        )

        if result.empty:
            continue

        for rank, (_, product) in enumerate(
            result.iterrows(),
            start=1
        ):

            search_annotation_rows.append({
                "query_id":
                    item["query_id"],

                "query":
                    item["query"],

                "rank":
                    rank,

                "product_id":
                    int(product["id"]),

                "title":
                    product["title_fa"],

                "category":
                    product.get("Category1"),

                "sub_category":
                    product.get("Category2"),

                "brand":
                    product.get("Brand"),

                "price":
                    product.get("Price"),

                "rating":
                    product.get("mean_rate"),

                "recommendation_ratio":
                    product.get("rec_ratio"),

                "search_score":
                    product.get("_final_score"),

                "relevant":
                    None,

                "annotation_note":
                    "",
            })

    search_annotation = pd.DataFrame(
        search_annotation_rows
    )

except Exception as error:

    print(
        "Search candidate generation skipped:",
        error
    )

    search_annotation = pd.DataFrame()


# ============================================================
# 7. COMPARISON QUERIES
# ============================================================

def build_comparison_pairs(
    products,
    n_pairs=15,
    min_reviews=10,
    random_state=42,
):
    df = products.copy()

    df["n_reviews_num"] = pd.to_numeric(
        df["n_reviews"],
        errors="coerce"
    ).fillna(0)

    df = df[
        df["n_reviews_num"] >= min_reviews
    ].copy()

    pairs = []

    if "Category2" not in df.columns:
        return pd.DataFrame()

    for category, group in (
        df.groupby(
            "Category2",
            dropna=True
        )
    ):

        if len(group) < 2:
            continue

        sampled = group.sample(
            n=min(
                10,
                len(group)
            ),
            random_state=random_state
        )

        for i in range(
            len(sampled) - 1
        ):

            a = sampled.iloc[i]
            b = sampled.iloc[i + 1]

            if int(a["id"]) == int(b["id"]):
                continue

            pairs.append({
                "product_id_a": int(a["id"]),
                "product_id_b": int(b["id"]),

                "title_a": a["title_fa"],
                "title_b": b["title_fa"],

                "category": category,

                "price_a": a["Price"],
                "price_b": b["Price"],

                "rating_a": a["mean_rate"],
                "rating_b": b["mean_rate"],

                "rec_ratio_a": a["rec_ratio"],
                "rec_ratio_b": b["rec_ratio"],
            })

            if len(pairs) >= n_pairs:
                return pd.DataFrame(pairs)

    return pd.DataFrame(pairs)


comparison_pairs = build_comparison_pairs(
    products,
    n_pairs=N_COMPARISON_PAIRS
)


comparison_templates = [
    "از نظر قیمت مقایسه کن.",
    "از نظر امتیاز و رضایت کاربران مقایسه کن.",
    "نقاط قوت و ضعف این دو محصول را بر اساس نظرات کاربران مقایسه کن.",
    "کدام یک ارزش خرید بیشتری دارد؟",
]


def build_comparison_queries(
    comparison_pairs
):
    rows = []

    for pair_index, pair in (
        comparison_pairs.iterrows()
    ):

        for template_index, template in enumerate(
            comparison_templates,
            start=1
        ):

            rows.append({
                "query_id":
                    f"comparison_{pair_index + 1}_{template_index}",

                "product_id_a":
                    int(pair["product_id_a"]),

                "product_id_b":
                    int(pair["product_id_b"]),

                "query":
                    (
                        f"محصول {int(pair['product_id_a'])} "
                        f"و محصول {int(pair['product_id_b'])} "
                        f"را بررسی کن. {template}"
                    ),

                "query_type":
                    "comparison",
            })

    return pd.DataFrame(rows)


comparison_queries = build_comparison_queries(
    comparison_pairs
)


# ============================================================
# 8. ANSWER QUALITY TEMPLATE
# ============================================================

# We create annotation rows now.
# Answers will be filled after running the final assistant.

all_answer_queries = pd.concat(
    [
        review_queries[
            [
                "query_id",
                "query"
            ]
        ],

        comparison_queries[
            [
                "query_id",
                "query"
            ]
        ],
    ],
    ignore_index=True
)

answer_quality = []

for _, row in all_answer_queries.iterrows():

    answer_quality.append({
        "query_id":
            row["query_id"],

        "query":
            row["query"],

        "answer":
            "",

        # Human annotation 1-5
        "relevance":
            None,

        "usefulness":
            None,

        "correctness":
            None,

        "completeness":
            None,

        "overall":
            None,

        "comment":
            "",
    })

answer_quality = pd.DataFrame(
    answer_quality
)


# ============================================================
# 9. GROUNDING TEMPLATE
# ============================================================

grounding = []

for _, row in all_answer_queries.iterrows():

    grounding.append({
        "query_id":
            row["query_id"],

        "query":
            row["query"],

        "answer":
            "",

        "claim":
            "",

        "claim_type":
            "",

        # 1 = supported
        # 0 = unsupported
        "supported":
            None,

        "supporting_review_ids":
            "",

        "supporting_product_fields":
            "",

        "comment":
            "",
    })

grounding = pd.DataFrame(
    grounding
)


# ============================================================
# 10. RECOMMENDATION DATASET
# ============================================================

recommendation = reviews[
    reviews[
        "recommendation_status"
    ].isin(
        [
            "recommended",
            "not_recommended",
        ]
    )
].copy()

recommendation["target"] = (
    recommendation[
        "recommendation_status"
    ]
    .map({
        "recommended": 1,
        "not_recommended": 0,
    })
)

recommendation_features = [
    "id",
    "product_id",
    "rate",
    "likes",
    "body_len",
    "is_buyer",
    "is_substantive",
    "target",
]

recommendation_features = [
    column
    for column in recommendation_features
    if column in recommendation.columns
]

recommendation = recommendation[
    recommendation_features
].copy()


# ============================================================
# 11. FAILURE ANALYSIS TEMPLATE
# ============================================================

all_query_ids = list(
    search_queries["query_id"]
) + list(
    review_queries["query_id"]
) + list(
    comparison_queries["query_id"]
)

failure_analysis = []

for query_id in all_query_ids:

    failure_analysis.append({
        "query_id":
            query_id,

        "failure":
            None,

        "failure_type":
            None,

        "cause":
            None,

        "retrieval_issue":
            None,

        "grounding_issue":
            None,

        "llm_issue":
            None,

        "latency_issue":
            None,

        "fix_attempt":
            None,

        "resolved":
            None,

        "comment":
            "",
    })

failure_analysis = pd.DataFrame(
    failure_analysis
)


# ============================================================
# 12. SAVE ALL DATASETS
# ============================================================

eval_products.to_csv(
    EVAL_DIR / "evaluation_products.csv",
    index=False,
    encoding="utf-8-sig"
)

eval_reviews.to_csv(
    EVAL_DIR / "evaluation_reviews.csv",
    index=False,
    encoding="utf-8-sig"
)

search_queries.to_csv(
    EVAL_DIR / "search_queries.csv",
    index=False,
    encoding="utf-8-sig"
)

search_annotation.to_csv(
    EVAL_DIR / "search_annotation.csv",
    index=False,
    encoding="utf-8-sig"
)

review_queries.to_csv(
    EVAL_DIR / "review_queries.csv",
    index=False,
    encoding="utf-8-sig"
)

review_annotation.to_csv(
    EVAL_DIR / "review_annotation.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_pairs.to_csv(
    EVAL_DIR / "comparison_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_queries.to_csv(
    EVAL_DIR / "comparison_queries.csv",
    index=False,
    encoding="utf-8-sig"
)

answer_quality.to_csv(
    EVAL_DIR / "answer_quality.csv",
    index=False,
    encoding="utf-8-sig"
)

grounding.to_csv(
    EVAL_DIR / "grounding.csv",
    index=False,
    encoding="utf-8-sig"
)

recommendation.to_parquet(
    EVAL_DIR / "recommendation_dataset.parquet",
    index=False
)

failure_analysis.to_csv(
    EVAL_DIR / "failure_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 13. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EVALUATION DATASET CREATED")
print("=" * 70)

print(
    "Evaluation products:",
    len(eval_products)
)

print(
    "Evaluation reviews:",
    len(eval_reviews)
)

print(
    "Search queries:",
    len(search_queries)
)

print(
    "Search annotation rows:",
    len(search_annotation)
)

print(
    "Review queries:",
    len(review_queries)
)

print(
    "Review annotation rows:",
    len(review_annotation)
)

print(
    "Comparison pairs:",
    len(comparison_pairs)
)

print(
    "Comparison queries:",
    len(comparison_queries)
)

print(
    "Answer quality rows:",
    len(answer_quality)
)

print(
    "Grounding rows:",
    len(grounding)
)

print(
    "Recommendation rows:",
    len(recommendation)
)

print(
    "Failure analysis rows:",
    len(failure_analysis)
)

print(
    "\nSaved to:",
    EVAL_DIR.resolve()
)

Products: (331599, 33)
Reviews : (6153060, 14)


e:\Quera_Proj_3\src\product_search.py:1069: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<ArrowExtensionArray>
[ 0.8500000059604644,                 0.0,  0.4950000029802322,  0.8871175338051276,   1.000000011920929,   0.931031617236464,  0.8500000059604644,  0.6766666750113168,  0.8411165204169093,
  0.7933136094962396,
 ...
  0.6416666636864343,  0.8239772715351799,  0.7215909212827682,    0.36458334227403,  0.9250000089406967, 0.15000000596046448,  0.4812500089406967,   0.689750641454821,  0.9625000178813934,
   0.609615396307065]
Length: 193, dtype: double[pyarrow]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  score.loc[
e:\Quera_Proj_3\src\product_search.py:1069: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '<ArrowExtensionArray>
[ 0.9648437619209289,  0.9000000178813934,


EVALUATION DATASET CREATED
Evaluation products: 30
Evaluation reviews: 205
Search queries: 28
Search annotation rows: 274
Review queries: 180
Review annotation rows: 1800
Comparison pairs: 15
Comparison queries: 60
Answer quality rows: 240
Grounding rows: 240
Recommendation rows: 4665272
Failure analysis rows: 268

Saved to: E:\evaluation
